In [1]:
import optuna

study_name = ""  # replace with your actual study name
dataset = "Simulations_indep_traincontrol"
n_trials = 150
optuna_version_name = "ExMetrics2_seedData{}_seedHPO{}".format(0, 11)
n_samples = 600
n_features_bytype = 6
treatment_effect = 0.
name_config = "simu_N{}_nfeat{}_t{}".format(n_samples, n_features_bytype, int(treatment_effect))
generator_name = "HI-VAE_piecewise" # "HI-VAE_weibull" # "HI-VAE_piecewise" 
study_name_cluster = "/home/pchassat/survgen-clinical-trials/dataset/{}/optuna_results/optuna_study_{}_ntrials{}_{}_{}".format(dataset, name_config, n_trials, optuna_version_name, generator_name)
db_file = "/Users/pchassat/Documents/survgen-clinical-trials/dataset/{}/optuna_results/optuna_study_{}_ntrials{}_{}_{}.db".format(dataset, name_config, n_trials, optuna_version_name, generator_name)
storage = f"sqlite:///{db_file}"
study = optuna.load_study(study_name=study_name_cluster, storage=storage)
names_objs = ["Survival curves dist","K-map score"]

/opt/anaconda3/envs/env_synthcity/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from optuna.visualization import plot_parallel_coordinate, plot_slice, plot_param_importances

for i in range(len(names_objs)):
    plot_parallel_coordinate(study, target=lambda t: t.values[i], target_name=names_objs[i]).show() # relationships between objectives and parameters

In [3]:
for i in range(len(names_objs)):
    plot_slice(study, target=lambda t: t.values[i], target_name=names_objs[i]).show() # individual parameter effects

In [5]:
for i in range(len(names_objs)):
    print(f"Parameter importances for {names_objs[i]}:")
    plot_param_importances(study, target=lambda t: t.values[i], target_name=names_objs[i]).show() 

Parameter importances for Survival curves dist:


Parameter importances for K-map score:


In [6]:
# Get all trials on the Pareto front
pareto_trials = study.best_trials  # these are Pareto optimal

for t in pareto_trials:
    print("Values:", t.values)
    print("Params:", t.params)
    print("------")

Values: [0.010150823551562481, -7.0]
Params: {'lr': 0.002, 'batch_size': 30, 'z_dim': 60, 'y_dim': 100, 's_dim': 180, 'n_layers_surv_piecewise': 2, 'n_intervals': 20}
------
Values: [0.017268786782999083, -11.0]
Params: {'lr': 0.002, 'batch_size': 40, 'z_dim': 60, 'y_dim': 100, 's_dim': 50, 'n_layers_surv_piecewise': 2, 'n_intervals': 20}
------
Values: [0.008343264188137505, -3.0]
Params: {'lr': 0.002, 'batch_size': 40, 'z_dim': 60, 'y_dim': 110, 's_dim': 190, 'n_layers_surv_piecewise': 1, 'n_intervals': 20}
------
Values: [0.014808061480923906, -9.0]
Params: {'lr': 0.002, 'batch_size': 80, 'z_dim': 30, 'y_dim': 50, 's_dim': 70, 'n_layers_surv_piecewise': 1, 'n_intervals': 20}
------
Values: [0.008772675184169106, -4.0]
Params: {'lr': 0.005, 'batch_size': 150, 'z_dim': 150, 'y_dim': 170, 's_dim': 180, 'n_layers_surv_piecewise': 2, 'n_intervals': 20}
------


In [7]:
import pandas as pd

# df_pareto = pd.DataFrame([
#     {**t.params, **{names_objs[i]: v for i, v in enumerate(t.values)}}
#     for t in study.best_trials
# ])
# print(df_pareto)

df_pareto = pd.DataFrame([
    {
        "trial_number": t.number,          
        **t.params,
        **{names_objs[i]: v for i, v in enumerate(t.values)}
    }
    for t in study.best_trials
])

df_pareto.head(10)

,trial_number,lr,batch_size,z_dim,y_dim,s_dim,n_layers_surv_piecewise,n_intervals,Survival curves dist,K-map score
0,48,0.002,30,60,100,180,2,20,0.010151,-7.0
1,60,0.002,40,60,100,50,2,20,0.017269,-11.0
2,102,0.002,40,60,110,190,1,20,0.008343,-3.0
3,104,0.002,80,30,50,70,1,20,0.014808,-9.0
4,124,0.005,150,150,170,180,2,20,0.008773,-4.0


In [8]:
from optuna.visualization import plot_optimization_history

for i in range(len(names_objs)):
    plot_optimization_history(study, target=lambda t: t.values[i], target_name=names_objs[i]).show() 

In [9]:
from optuna.trial import TrialState
completed = [t for t in study.trials if t.state == TrialState.COMPLETE]
print("Number of trials:", len(study.trials))
print("Number of completed trials:", len(completed))

Number of trials: 150
Number of completed trials: 150


In [10]:
from optuna.visualization import plot_pareto_front
plot_pareto_front(
    study,
    targets=lambda t: [t.values[0], t.values[1]],
    target_names=["Survival curves dist", "K-map score"],
    include_dominated_trials=True
)

In [11]:
selected_trial_id = 60
best_trial = study.trials[selected_trial_id]

print("Values (objectifs):", best_trial.values)
print("Params:", best_trial.params)
print("State:", best_trial.state)
print("Start:", best_trial.datetime_start)
print("End:", best_trial.datetime_complete)
print("Duration:", best_trial.duration)

Values (objectifs): [0.017268786782999083, -11.0]
Params: {'lr': 0.002, 'batch_size': 40, 'z_dim': 60, 'y_dim': 100, 's_dim': 50, 'n_layers_surv_piecewise': 2, 'n_intervals': 20}
State: 1
Start: 2026-06-10 02:49:19.372958
End: 2026-06-10 02:51:40.755300
Duration: 0:02:21.382342


### Save best selected trial

In [11]:
import json
parent_path = "/Users/pchassat/Documents/survgen-clinical-trials"
best_params_file = parent_path + "/dataset/" + dataset + "/optuna_results/best_params_{}_ntrials{}_{}_{}.json".format(name_config, n_trials, optuna_version_name, generator_name)
with open(best_params_file, "w") as f:
    json.dump(best_trial.params, f)